In [ ]:
import os, sys, json, time, hashlib, math
import numpy as np
import pandas as pd
import torch, torch.nn as nn
from scipy.io import wavfile
from tqdm.auto import tqdm
from sklearn.model_selection import StratifiedGroupKFold
from transformers import AutoModel, AutoFeatureExtractor, get_linear_schedule_with_warmup


In [ ]:
BASE_DIR = os.environ.get("MELD_BASE_DIR", "/kaggle/working")
CONFIG_PATH = os.path.join(BASE_DIR, "config.json")
config = json.load(open(CONFIG_PATH))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR, FEATURES_DIR, RESULTS_DIR, WAV_DIR = (config["paths"][k] for k in ("data_dir", "features_dir", "results_dir", "audio_wav_dir"))
EMOTIONS, C = config["emotions"], len(config["emotions"])
SPLITS = ["train", "dev", "test"]
ENABLED = os.environ.get("MELD_AUDIO_FT", "1").strip().lower() in ("1", "true", "yes")
K = int(os.environ.get("MELD_P3B_FOLDS", 5))                    
EPOCHS = int(os.environ.get("MELD_P4B_EPOCHS", 4))                
LR = float(os.environ.get("MELD_P4B_LR", 3e-5))
BATCH = int(os.environ.get("MELD_P4B_BATCH", 16))
MAX_SEC = float(os.environ.get("MELD_P4B_MAXSEC", 8.0))
WEIGHT_POWER = float(os.environ.get("MELD_P4B_WEIGHT_POWER", 0.5))
SEED = int(config["seed"])
MODEL_ID = config.get("audio_model", "microsoft/wavlm-base-plus")
SR = int(config["audio_sample_rate"])
FT_FILE = "audio_ft_features_{split}.npy"
META_PATH, REPORT_PATH, VERIF_PATH = os.path.join(FEATURES_DIR, "audio_ft_meta.json"), os.path.join(RESULTS_DIR, "audio_ft_report.json"), os.path.join(RESULTS_DIR, "audio_ft_verification.json")
CACHE_DIR = os.path.join(FEATURES_DIR, "_p4b_folds"); os.makedirs(CACHE_DIR, exist_ok=True)
SETTINGS = {"model": MODEL_ID, "folds": K, "epochs": EPOCHS, "lr": LR, "batch": BATCH, "max_sec": MAX_SEC, "weight_power": WEIGHT_POWER, "seed": SEED,
            "pooling": "mean_over_valid_frames", "freeze": "conv_feature_encoder", "input": "centre-crop, per-clip normalisation, length-bucketed batches",
            "dropout": 0.1, "warmup": 0.1, "weight_decay": 0.01}
SETTINGS_ID = hashlib.sha1(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:10]

def write_json(path, obj):
    tmp = path + ".tmp"
    with open(tmp, "w") as f:
        json.dump(obj, f, indent=2)
    os.replace(tmp, path)

if not ENABLED:
    for key in ("audio_feature_file", "audio_fusion_feature_dim", "audio_ft_feature_set"):
        config.pop(key, None)
    write_json(CONFIG_PATH, config)
    write_json(META_PATH, {"enabled": False})
    write_json(REPORT_PATH, {"enabled": False, "note": "MELD_AUDIO_FT=0: the frozen Phase 4 audio features are used"})
    write_json(VERIF_PATH, {"n_tests": 0, "n_failed": 0, "results": []})
    print("MELD_AUDIO_FT=0: fine-tuned audio features are OFF; config.json points to the frozen Phase 4 features. Remaining cells are skipped.")
else:
    print(f"Device: {device} | model: {MODEL_ID} | folds: {K} | epochs: {EPOCHS} | lr: {LR} | batch: {BATCH} | max {MAX_SEC}s | settings id: {SETTINGS_ID}")


In [ ]:
def assign_folds(labels, groups, k, seed):
    fold = np.full(len(labels), -1, dtype=np.int64)
    for f, (_, ho) in enumerate(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(np.zeros(len(labels)), labels, groups)):
        fold[ho] = f
    return fold

def log_softmax_np(z):
    z = np.asarray(z, dtype=np.float64); z = z - z.max(1, keepdims=True)
    return z - np.log(np.exp(z).sum(1, keepdims=True))

def zscore_fit(x):
    mu, sd = x.mean(0), x.std(0)
    return mu, np.where(sd < 1e-8, 1.0, sd)

def class_weights_np(labels, c, power):
    cnt = np.bincount(labels, minlength=c).astype(np.float64)
    w = (len(labels) / (c * np.maximum(cnt, 1))) ** power
    return w / (w[labels]).mean()

def macro_f1_np(y, p, c):
    conf = np.bincount(np.asarray(y) * c + np.asarray(p), minlength=c * c).reshape(c, c).astype(np.float64)
    tp, row, col = np.diag(conf), conf.sum(1), conf.sum(0)
    return float(np.where(row + col > 0, 2 * tp / np.maximum(row + col, 1e-300), 0.0).mean())

def read_wav_mono_f32(path):
    sr, x = wavfile.read(path)
    if np.issubdtype(x.dtype, np.integer):
        x = x.astype(np.float32) / float(np.iinfo(x.dtype).max + 1)
    else:
        x = x.astype(np.float32)
    if x.ndim > 1:
        x = x.mean(axis=1).astype(np.float32)
    return x, int(sr)

def prepare_wave(x, max_len):
    """Centre-crop to max_len samples, then zero-mean / unit-variance per clip (what the WavLM feature extractor does)."""
    x = np.asarray(x, dtype=np.float32)
    if len(x) > max_len:
        s = (len(x) - max_len) // 2
        x = x[s:s + max_len]
    return ((x - x.mean()) / np.sqrt(x.var() + 1e-7)).astype(np.float32)

def bucket_batches(lengths, idx, batch, rng, mega=50):
    """Shuffle idx, cut into chunks of mega*batch, sort each chunk by length, split into batches (batches are then shuffled): little padding."""
    idx = np.asarray(idx); perm = idx[rng.permutation(len(idx))]
    out = []
    for i in range(0, len(perm), mega * batch):
        chunk = perm[i:i + mega * batch]
        chunk = chunk[np.argsort([lengths[j] for j in chunk], kind="stable")]
        out += [chunk[b:b + batch] for b in range(0, len(chunk), batch)]
    return [out[i] for i in rng.permutation(len(out))]

def pad_batch(waves):
    lens = np.array([len(w) for w in waves], dtype=np.int64)
    arr = np.zeros((len(waves), int(lens.max())), dtype=np.float32)
    for i, w in enumerate(waves):
        arr[i, :len(w)] = w
    return arr, lens
print("Helpers defined.")


In [ ]:
if ENABLED:
    dfs = {s: pd.read_csv(os.path.join(DATA_DIR, f"{s}_sent_emo.csv"), keep_default_na=False) for s in SPLITS}
    mans = {s: pd.read_csv(os.path.join(DATA_DIR, f"media_manifest_{s}.csv"), keep_default_na=False) for s in SPLITS}
    emo2id = {e: i for i, e in enumerate(EMOTIONS)}
    labels = {s: dfs[s]["Emotion"].str.lower().map(emo2id).values for s in SPLITS}
    for s in SPLITS:
        assert not pd.isnull(labels[s]).any(), f"[{s}] unknown emotion labels"
        labels[s] = labels[s].astype(np.int64)
        assert len(mans[s]) == len(dfs[s]) and (mans[s]["row_idx"].values == np.arange(len(dfs[s]))).all(), f"[{s}] manifest/CSV misalignment"
    valid = {s: np.load(os.path.join(FEATURES_DIR, f"audio_valid_{s}.npy")).astype(bool) for s in SPLITS}
    frozen = {s: np.load(os.path.join(FEATURES_DIR, f"audio_features_{s}.npy")) for s in SPLITS}
    for s in SPLITS:
        assert frozen[s].shape == (len(dfs[s]), config["audio_feature_dim"]) and valid[s].shape == (len(dfs[s]),), f"[{s}] Phase 4 outputs have unexpected shapes -- run Phase 4 first"
    groups_train = dfs["train"]["Dialogue_ID"].values
    FOLD = assign_folds(labels["train"], groups_train, K, SEED)
    MAX_LEN = int(MAX_SEC * SR)
    WAVE = {}
    for s in SPLITS:
        WAVE[s] = [None] * len(dfs[s])
        for i in tqdm(np.flatnonzero(valid[s]), desc=f"loading wavs ({s})"):
            x, sr = read_wav_mono_f32(os.path.join(WAV_DIR, s, os.path.splitext(mans[s].at[i, "video_file"])[0] + ".wav"))
            assert sr == SR, f"[{s}] row {i}: sample rate {sr} != {SR}"
            WAVE[s][i] = prepare_wave(x, MAX_LEN)
    LENS = {s: np.array([0 if w is None else len(w) for w in WAVE[s]]) for s in SPLITS}
    print("usable clips:", {s: int(valid[s].sum()) for s in SPLITS}, "| fold sizes (utterances):", np.bincount(FOLD).tolist())
    print("mean clip length (s):", {s: round(float(LENS[s][valid[s]].mean() / SR), 2) for s in SPLITS})


In [ ]:
if ENABLED:
    extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)
    USE_ATT = bool(getattr(extractor, "return_attention_mask", False))          
    class AudioClassifier(nn.Module):
        def __init__(self):
            super().__init__()
            self.enc = AutoModel.from_pretrained(MODEL_ID)
            self.enc.freeze_feature_encoder()                                    
            self.drop = nn.Dropout(0.1)
            self.head = nn.Linear(self.enc.config.hidden_size, C)
        def forward(self, wave, lens):
            T = wave.shape[1]
            att = (torch.arange(T, device=wave.device)[None, :] < lens[:, None]).long()
            h = self.enc(wave, attention_mask=att if USE_ATT else None).last_hidden_state
            flen = self.enc._get_feat_extract_output_lengths(lens).clamp(min=1, max=h.shape[1])
            fm = (torch.arange(h.shape[1], device=wave.device)[None, :] < flen[:, None]).unsqueeze(-1).to(h.dtype)
            return self.head(self.drop((h * fm).sum(1) / fm.sum(1).clamp(min=1.0)))     

    def to_batch(split, rows):
        arr, lens = pad_batch([WAVE[split][j] for j in rows])
        return torch.from_numpy(arr).to(device), torch.from_numpy(lens).to(device)

    @torch.no_grad()
    def predict(model, split, rows, bs=32):
        model.eval(); rows = np.asarray(rows); order = rows[np.argsort(LENS[split][rows], kind="stable")]
        out = {}
        for i in range(0, len(order), bs):
            sel = order[i:i + bs]; w, l = to_batch(split, sel)
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=(device.type == "cuda")):
                lg = model(w, l).float().cpu().numpy()
            for j, r in enumerate(sel):
                out[int(r)] = lg[j]
        return np.stack([out[int(r)] for r in rows]) if len(rows) else np.zeros((0, C), np.float32)

    def train_fold(f):
        tr = np.flatnonzero((FOLD != f) & valid["train"]); ho = np.flatnonzero((FOLD == f) & valid["train"])
        assert len(set(groups_train[tr]) & set(groups_train[ho])) == 0, "a dialogue is in both the training part and the held-out part"
        torch.manual_seed(SEED + f); np.random.seed(SEED + f)
        model = AudioClassifier().to(device)
        w = torch.tensor(class_weights_np(labels["train"][tr], C, WEIGHT_POWER), dtype=torch.float32, device=device)
        lossf = nn.CrossEntropyLoss(weight=w)
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=LR, weight_decay=0.01)
        steps = EPOCHS * math.ceil(len(tr) / BATCH)
        sched = get_linear_schedule_with_warmup(opt, int(0.1 * steps), steps)
        use_amp = device.type == "cuda"
        scaler = torch.amp.GradScaler(device.type, enabled=use_amp)
        ytr = torch.tensor(labels["train"], dtype=torch.long)
        for ep in range(EPOCHS):
            model.train(); rng = np.random.RandomState(SEED * 1000 + f * 10 + ep); tot, n = 0.0, 0
            for sel in tqdm(bucket_batches(LENS["train"], tr, BATCH, rng), desc=f"fold {f} epoch {ep + 1}/{EPOCHS}", leave=False):
                wv, ln = to_batch("train", sel); y = ytr[sel].to(device)
                with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                    logits = model(wv, ln)
                loss = lossf(logits.float(), y)
                opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); scaler.step(opt); scaler.update(); sched.step()
                tot += float(loss) * len(sel); n += len(sel)
            print(f"  fold {f} epoch {ep + 1}: train loss {tot / n:.4f}")
        dv, ts = np.flatnonzero(valid["dev"]), np.flatnonzero(valid["test"])
        res = {"ho_idx": ho, "ho_logits": predict(model, "train", ho), "dev_idx": dv, "dev_logits": predict(model, "dev", dv), "test_idx": ts, "test_logits": predict(model, "test", ts)}
        del model, opt; torch.cuda.empty_cache()
        return res
    print("Model / training functions defined.")


In [ ]:
if ENABLED:
    FOLD_RES, t_start = {}, time.time()
    for f in range(K):
        cache = os.path.join(CACHE_DIR, f"fold{f}_{SETTINGS_ID}.npz")
        if os.path.exists(cache):
            z = np.load(cache); FOLD_RES[f] = {k: z[k] for k in ("ho_idx", "ho_logits", "dev_idx", "dev_logits", "test_idx", "test_logits")}
            print(f"[fold {f}] found cached result ({cache}) -> skipped"); continue
        t0 = time.time(); FOLD_RES[f] = train_fold(f)
        np.savez(cache + ".tmp.npz", **FOLD_RES[f]); os.replace(cache + ".tmp.npz", cache)
        ho = FOLD_RES[f]["ho_idx"]; p = FOLD_RES[f]["ho_logits"].argmax(1); y = labels["train"][ho]
        dv = FOLD_RES[f]["dev_idx"]
        print(f"[fold {f}] {(time.time() - t0) / 60:.1f} min | held-out acc {(p == y).mean():.4f} macro-F1 {macro_f1_np(y, p, C):.4f} | dev acc {(FOLD_RES[f]['dev_logits'].argmax(1) == labels['dev'][dv]).mean():.4f}")
    print(f"All folds done in {(time.time() - t_start) / 60:.1f} min")


In [ ]:
if ENABLED:
    n_tr = len(labels["train"])
    oof = np.zeros((n_tr, C), dtype=np.float64); seen = np.zeros(n_tr, dtype=int)
    for f in range(K):
        oof[FOLD_RES[f]["ho_idx"]] = log_softmax_np(FOLD_RES[f]["ho_logits"]); seen[FOLD_RES[f]["ho_idx"]] += 1
    assert (seen[valid["train"]] == 1).all() and (seen[~valid["train"]] == 0).all(), "every USABLE training clip must receive exactly one out-of-fold prediction (unusable clips none)"
    lp = {"train": oof}
    for s in ("dev", "test"):
        acc = np.zeros((len(dfs[s]), C), dtype=np.float64)
        for f in range(K):
            acc[FOLD_RES[f][f"{s}_idx"]] += log_softmax_np(FOLD_RES[f][f"{s}_logits"])
        lp[s] = acc / K
    vt = valid["train"]
    MU, SD = zscore_fit(lp["train"][vt])                 
    FT = {}
    for s in SPLITS:
        z = ((lp[s] - MU) / SD).astype(np.float32); z[~valid[s]] = 0.0
        FT[s] = np.concatenate([frozen[s], z], axis=1).astype(np.float32)
    DIM = int(FT["train"].shape[1])
    pred = {s: lp[s].argmax(1) for s in SPLITS}
    rep = {"settings": SETTINGS, "settings_id": SETTINGS_ID, "dim": DIM, "frozen_dim": int(config["audio_feature_dim"]), "logit_dims": C,
           "n_usable": {s: int(valid[s].sum()) for s in SPLITS},
           "oof_train": {"accuracy": float((pred["train"][vt] == labels["train"][vt]).mean()), "macro_f1": macro_f1_np(labels["train"][vt], pred["train"][vt], C)},
           "dev_kmodel_average": {"accuracy": float((pred["dev"][valid["dev"]] == labels["dev"][valid["dev"]]).mean()), "macro_f1": macro_f1_np(labels["dev"][valid["dev"]], pred["dev"][valid["dev"]], C)},
           "per_fold_dev_macro_f1": [macro_f1_np(labels["dev"][FOLD_RES[f]["dev_idx"]], FOLD_RES[f]["dev_logits"].argmax(1), C) for f in range(K)],
           "train_majority_rate": float(np.bincount(labels["train"][vt], minlength=C).max() / vt.sum()),
           "note": "dev/test rows get the K-model average; unusable clips have 0 in the new 7 dims and stay flagged unusable; test labels are never read in this notebook"}
    print(json.dumps(rep, indent=2))


In [ ]:
if ENABLED:
    res = []
    def check(name, cond, detail=""):
        res.append({"test": name, "passed": bool(cond), "detail": str(detail)}); print(("PASS  " if cond else "FAIL  ") + name + (f"  [{detail}]" if detail else ""))
    ho_sets = [set(FOLD_RES[f]["ho_idx"].tolist()) for f in range(K)]
    check("W1 held-out parts are disjoint and cover every USABLE training clip once", sum(len(h) for h in ho_sets) == int(vt.sum()) and len(set().union(*ho_sets)) == int(vt.sum()) and set().union(*ho_sets) == set(np.flatnonzero(vt).tolist()))
    check("W2 no dialogue appears in two folds", all(len(set(groups_train[FOLD == a]) & set(groups_train[FOLD == b])) == 0 for a in range(K) for b in range(a + 1, K)))
    check("W3 every fold model was trained without its held-out dialogues (train_fold asserts this per fold)", True, "enforced by an assert inside train_fold")
    check("W4 out-of-fold log-probabilities of usable clips are valid", np.isfinite(oof[vt]).all() and np.allclose(np.exp(oof[vt]).sum(1), 1.0, atol=1e-6))
    check("W5 fine-tuning worked: OOF macro-F1 clearly above a constant predictor", rep["oof_train"]["macro_f1"] > 0.15, f"OOF macro-F1 {rep['oof_train']['macro_f1']:.3f} (constant predictor ~0.09)")
    check("W6 fine-tuned signal is not trivially perfect (would indicate leakage)", rep["oof_train"]["accuracy"] < 0.90 and rep["dev_kmodel_average"]["accuracy"] < 0.90)
    check("W7 frozen part of the output equals the Phase 4 file exactly", all(np.array_equal(FT[s][:, :config["audio_feature_dim"]], frozen[s]) for s in SPLITS))
    zt = FT["train"][vt][:, config["audio_feature_dim"]:]
    check("W8 z-scored log-probabilities have mean 0 / std 1 on the usable training clips", np.allclose(zt.mean(0), 0, atol=1e-4) and np.allclose(zt.std(0), 1, atol=1e-3))
    check("W9 all features finite; shapes match the CSV rows", all(np.isfinite(FT[s]).all() and FT[s].shape == (len(dfs[s]), DIM) for s in SPLITS), DIM)
    check("W10 unusable clips have exactly 0 in the new part (and keep the Phase 4 value in the frozen part)", all(np.all(FT[s][~valid[s]][:, config["audio_feature_dim"]:] == 0) for s in SPLITS))
    check("W11 dev / test predictions exist for every usable clip", all(all(set(FOLD_RES[f][f"{s}_idx"].tolist()) == set(np.flatnonzero(valid[s]).tolist()) for f in range(K)) for s in ("dev", "test")))
    check("W12 no clip is longer than the crop", all(int(LENS[s].max()) <= MAX_LEN for s in SPLITS))
    failed = [r for r in res if not r["passed"]]
    write_json(VERIF_PATH, {"n_tests": len(res), "n_failed": len(failed), "results": res})
    assert not failed, f"{len(failed)} check(s) failed: {[r['test'] for r in failed]}  -- config.json and the feature files were NOT changed"
    for s in SPLITS:
        path = os.path.join(FEATURES_DIR, FT_FILE.format(split=s)); tmp = path + ".tmp.npy"
        np.save(tmp, FT[s]); os.replace(tmp, path)
    def sha1_file(p):
        h = hashlib.sha1()
        with open(p, "rb") as fh:
            for chunk in iter(lambda: fh.read(1 << 20), b""):
                h.update(chunk)
        return h.hexdigest()
    write_json(META_PATH, {"enabled": True, "dim": DIM, "settings": SETTINGS, "settings_id": SETTINGS_ID, "z_mu": MU.tolist(), "z_sd": SD.tolist(),
                           "frozen_sha1": {s: sha1_file(os.path.join(FEATURES_DIR, f"audio_features_{s}.npy")) for s in SPLITS}})
    write_json(REPORT_PATH, rep)
    config = json.load(open(CONFIG_PATH))
    config.update({"audio_feature_file": FT_FILE, "audio_fusion_feature_dim": DIM,
                   "audio_ft_feature_set": f"frozen {MODEL_ID} uniform-layer mean + out-of-fold fine-tuned {MODEL_ID} log-probabilities ({K} dialogue-grouped folds, {EPOCHS} epochs, fixed)"})
    write_json(CONFIG_PATH, config)
    print(f"\nAll {len(res)} checks passed. Saved {[FT_FILE.format(split=s) for s in SPLITS]} ({DIM}-d) and updated config.json.")
    print("Phase 4b Fine-tuned audio features (out-of-fold) completed successfully. Phase 6 onward uses them automatically.")
